In [1]:
# sinh con , check goal, insert queue
import tkinter as tk
from collections import deque
import random
import time

GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)
TILE = 90
GAP = 5
ANIM = 300
def neighbors(state):
    idx = state.index(0)
    r, c = divmod(idx, 3)
    result = []
    for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            ni = nr * 3 + nc
            temp = list(state)
            temp[idx], temp[ni] = temp[ni], temp[idx]
            result.append(tuple(temp))
    return result
def bfs_v1(start):
    if start == GOAL:
        return [start], 0
    visited = {start}
    queue = deque([(start, [start])])
    nodes = 0
    while queue:
        state, path = queue.popleft()
        nodes += 1
        for child in neighbors(state):
            if child in visited:
                continue
            if child == GOAL:  # check goal
                return path + [child], nodes
            visited.add(child)
            queue.append((child, path + [child]))
    return None, nodes

def is_solvable(board):
    arr = [x for x in board if x != 0]
    inv = 0
    for i in range(len(arr)):
        for j in range(i + 1, len(arr)):
            if arr[i] > arr[j]:
                inv += 1
    return inv % 2 == 0
def random_board():
    b = list(GOAL)
    while True:
        random.shuffle(b)
        t = tuple(b)
        if is_solvable(t) and t != GOAL:
            return t
class App:
    def __init__(self, root):
        self.root = root
        self.root.title("8 Puzzle BFS")
        self.root.configure(bg="white")
        self.root.resizable(False, False)
        self.board = random_board()
        self.solution = []
        self.step = 0
        self.playing = False
        self.canvas = tk.Canvas(
            root,
            width=3*TILE + 4*GAP,
            height=3*TILE + 4*GAP,
            bg="white",
            highlightthickness=0
        )
        self.canvas.pack(pady=10)
        info = tk.Frame(root, bg="white")
        info.pack()
        self.lb_nodes = tk.Label(info, text="Nodes: 0", bg="white")
        self.lb_nodes.grid(row=0, column=0, padx=10)
        self.lb_steps = tk.Label(info, text="Steps: 0", bg="white")
        self.lb_steps.grid(row=0, column=1, padx=10)
        self.lb_time = tk.Label(info, text="Time: 0 ms", bg="white")
        self.lb_time.grid(row=0, column=2, padx=10)
        btns = tk.Frame(root, bg="white")
        btns.pack(pady=10)
        tk.Button(
            btns,
            text="Shuffle",
            width=10,
            command=self.shuffle
        ).grid(row=0, column=0, padx=5)

        tk.Button(
            btns,
            text="Solve",
            width=10,
            command=self.solve
        ).grid(row=0, column=1, padx=5)

        tk.Button(
            btns,
            text="Step",
            width=10,
            command=self.next_step
        ).grid(row=0, column=2, padx=5)

        self.draw()
    def current_board(self):
        if self.solution:
            return self.solution[self.step]
        return self.board
    def draw(self):
        self.canvas.delete("all")
        board = self.current_board()
        for i, val in enumerate(board):
            r, c = divmod(i, 3)
            x1 = GAP + c * (TILE + GAP)
            y1 = GAP + r * (TILE + GAP)
            x2 = x1 + TILE
            y2 = y1 + TILE
            self.canvas.create_rectangle(
                x1, y1, x2, y2,
                fill="white",
                outline="black",
                width=2
            )
            if val != 0:
                self.canvas.create_text(
                    (x1+x2)//2,
                    (y1+y2)//2,
                    text=str(val),
                    font=("Arial", 28),
                    fill="black"
                )
    def shuffle(self):
        self.playing = False
        self.board = random_board()
        self.solution = []
        self.step = 0
        self.lb_nodes.config(text="Nodes: 0")
        self.lb_steps.config(text="Steps: 0")
        self.lb_time.config(text="Time: 0 ms")
        self.draw()
    def solve(self):
        if self.playing:
            return
        t0 = time.perf_counter()
        path, nodes = bfs_v1(self.board)
        ms = (time.perf_counter() - t0) * 1000
        if path is None:
            return
        self.solution = path
        self.step = 0
        self.lb_nodes.config(text=f"Nodes: {nodes}")
        self.lb_steps.config(text=f"Steps: {len(path)-1}")
        self.lb_time.config(text=f"Time: {ms:.2f} ms")
        self.playing = True
        self.animate()
    def animate(self):
        if not self.playing:
            return
        self.draw()
        if self.step < len(self.solution) - 1:
            self.step += 1
            self.root.after(ANIM, self.animate)
        else:
            self.playing = False
    def next_step(self):
        self.playing = False
        if not self.solution:
            return
        if self.step < len(self.solution) - 1:
            self.step += 1
            self.draw()
if __name__ == "__main__":
    root = tk.Tk()
    app = App(root)
    root.mainloop()

In [2]:
#sinh con ,insert queue, pop ra, check goal
import tkinter as tk
from collections import deque
import random
import time

GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)
TILE = 90
GAP = 5
ANIM = 300
def get_neighbors(board):
    idx = board.index(0)
    r, c = divmod(idx, 3)
    result = []
    for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            ni = nr * 3 + nc
            temp = list(board)
            temp[idx], temp[ni] = temp[ni], temp[idx]
            result.append(tuple(temp))
    return result

def bfs_v2(start):
    reached = {start}
    queue = deque([(start, [start])])
    nodes = 0
    while queue:
        board, path = queue.popleft()
        nodes += 1
        if board == GOAL: # pop ra roi check goal
            return path, nodes
        for child in get_neighbors(board):
            if child not in reached:
                reached.add(child)
                queue.append((child, path + [child])) # insert queue
    return None, nodes

def is_solvable(board):
    arr = [x for x in board if x != 0]
    inv = 0
    for i in range(len(arr)):
        for j in range(i + 1, len(arr)):
            if arr[i] > arr[j]:
                inv += 1
    return inv % 2 == 0

def random_board():
    b = list(GOAL)
    while True:
        random.shuffle(b)
        t = tuple(b)
        if is_solvable(t) and t != GOAL:
            return t

class App:
    def __init__(self, root):
        self.root = root
        self.root.title("8 Puzzle BFS V2")
        self.root.configure(bg="white")
        self.root.resizable(False, False)
        self.board = random_board()
        self.solution = []
        self.step = 0
        self.playing = False
        self.canvas = tk.Canvas(
            root,
            width=3*TILE + 4*GAP,
            height=3*TILE + 4*GAP,
            bg="white",
            highlightthickness=0
        )
        self.canvas.pack(pady=10)
        info = tk.Frame(root, bg="white")
        info.pack()
        self.lb_nodes = tk.Label(info, text="Nodes: 0", bg="white")
        self.lb_nodes.grid(row=0, column=0, padx=10)
        self.lb_steps = tk.Label(info, text="Steps: 0", bg="white")
        self.lb_steps.grid(row=0, column=1, padx=10)
        self.lb_time = tk.Label(info, text="Time: 0 ms", bg="white")
        self.lb_time.grid(row=0, column=2, padx=10)
        btns = tk.Frame(root, bg="white")
        btns.pack(pady=10)

        tk.Button(
            btns,
            text="Shuffle",
            width=10,
            command=self.shuffle
        ).grid(row=0, column=0, padx=5)

        tk.Button(
            btns,
            text="Reset",
            width=10,
            command=self.reset
        ).grid(row=0, column=1, padx=5)

        tk.Button(
            btns,
            text="Solve",
            width=10,
            command=self.solve
        ).grid(row=0, column=2, padx=5)

        tk.Button(
            btns,
            text="Step",
            width=34,
            command=self.next_step
        ).grid(row=1, column=0, columnspan=3, pady=5)
        self.draw()

    def current_board(self):
        if self.solution:
            return self.solution[self.step]
        return self.board

    def draw(self):
        self.canvas.delete("all")
        board = self.current_board()
        for i, val in enumerate(board):
            r, c = divmod(i, 3)
            x1 = GAP + c * (TILE + GAP)
            y1 = GAP + r * (TILE + GAP)
            x2 = x1 + TILE
            y2 = y1 + TILE
            self.canvas.create_rectangle(
                x1,
                y1,
                x2,
                y2,
                fill="white",
                outline="black",
                width=2
            )
            if val != 0:
                self.canvas.create_text(
                    (x1+x2)//2,
                    (y1+y2)//2,
                    text=str(val),
                    font=("Arial", 28),
                    fill="black"
                )
    def shuffle(self):
        self.playing = False
        self.board = random_board()
        self.solution = []
        self.step = 0
        self.lb_nodes.config(text="Nodes: 0")
        self.lb_steps.config(text="Steps: 0")
        self.lb_time.config(text="Time: 0 ms")
        self.draw()

    def reset(self):
        self.playing = False
        self.board = GOAL
        self.solution = []
        self.step = 0
        self.lb_nodes.config(text="Nodes: 0")
        self.lb_steps.config(text="Steps: 0")
        self.lb_time.config(text="Time: 0 ms")
        self.draw()

    def solve(self):
        if self.playing:
            return
        t0 = time.perf_counter()
        path, nodes = bfs_v2(self.board)
        ms = (time.perf_counter() - t0) * 1000
        if path is None:
            return
        self.solution = path
        self.step = 0
        self.lb_nodes.config(text=f"Nodes: {nodes}")
        self.lb_steps.config(text=f"Steps: {len(path)-1}")
        self.lb_time.config(text=f"Time: {ms:.2f} ms")
        self.playing = True
        self.animate()

    def animate(self):
        if not self.playing:
            return
        self.draw()
        if self.step < len(self.solution) - 1:
            self.step += 1
            self.root.after(ANIM, self.animate)
        else:
            self.playing = False

    def next_step(self):
        self.playing = False
        if not self.solution:
            return
        if self.step < len(self.solution) - 1:
            self.step += 1
            self.draw()

if __name__ == "__main__":
    root = tk.Tk()
    app = App(root)
    root.mainloop()